# Trial Task notebook

## Charles Dezons

### charles_dezons@berkeley.edu

#### Let's first convert timestamps to a uniform interval (e.g. 1-minute bins like in the paper) for aggregation. In this example, we group order book updates by symbol and by each one-minute interval to prepare for feature calculation:

In [36]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('first_25000_rows.csv')

# Ensure timestamps are in datetime and set index
df['ts_event'] = pd.to_datetime(df['ts_event'])
df.set_index('ts_event', inplace=True)

# Group into 1-minute intervals per symbol
df_1min = df.groupby(['symbol', pd.Grouper(freq='1Min')]).last().reset_index()
print(df_1min[['symbol','ts_event']].head())  # show symbol and time of binned data

  symbol                  ts_event
0   AAPL 2024-10-21 11:54:00+00:00
1   AAPL 2024-10-21 11:55:00+00:00
2   AAPL 2024-10-21 11:56:00+00:00
3   AAPL 2024-10-21 11:57:00+00:00
4   AAPL 2024-10-21 11:58:00+00:00


We take the last order book state in each 1-minute window for each stock (using .last() after grouping) to represent the state at the end of that minute. This will be used, along with the order flow within that minute, to compute the imbalance features. The resulting df_1min has one row per symbol per minute.

In [37]:
df.head()

,ts_recv,rtype,publisher_id,instrument_id,action,side,depth,price,size,flags,...,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
ts_event,,,,,,,,,,,,,,,,,,,,,
2024-10-21 11:54:29.221064336+00:00,2024-10-21T11:54:29.221230963Z,10,2,38,C,B,1,233.62,2,130,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
2024-10-21 11:54:29.223769812+00:00,2024-10-21T11:54:29.223936626Z,10,2,38,A,B,0,233.67,2,130,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
2024-10-21 11:54:29.225030400+00:00,2024-10-21T11:54:29.225196809Z,10,2,38,A,B,0,233.67,3,130,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
2024-10-21 11:54:29.712434212+00:00,2024-10-21T11:54:29.712600612Z,10,2,38,A,B,2,233.52,200,130,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
2024-10-21 11:54:29.764673165+00:00,2024-10-21T11:54:29.764839221Z,10,2,38,C,B,2,233.52,200,130,...,155,1,7,233.25,234.13,55,400,2,1,AAPL


In [38]:
df.columns

Index(['ts_recv', 'rtype', 'publisher_id', 'instrument_id', 'action', 'side',
       'depth', 'price', 'size', 'flags', 'ts_in_delta', 'sequence',
       'bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00', 'bid_ct_00',
       'ask_ct_00', 'bid_px_01', 'ask_px_01', 'bid_sz_01', 'ask_sz_01',
       'bid_ct_01', 'ask_ct_01', 'bid_px_02', 'ask_px_02', 'bid_sz_02',
       'ask_sz_02', 'bid_ct_02', 'ask_ct_02', 'bid_px_03', 'ask_px_03',
       'bid_sz_03', 'ask_sz_03', 'bid_ct_03', 'ask_ct_03', 'bid_px_04',
       'ask_px_04', 'bid_sz_04', 'ask_sz_04', 'bid_ct_04', 'ask_ct_04',
       'bid_px_05', 'ask_px_05', 'bid_sz_05', 'ask_sz_05', 'bid_ct_05',
       'ask_ct_05', 'bid_px_06', 'ask_px_06', 'bid_sz_06', 'ask_sz_06',
       'bid_ct_06', 'ask_ct_06', 'bid_px_07', 'ask_px_07', 'bid_sz_07',
       'ask_sz_07', 'bid_ct_07', 'ask_ct_07', 'bid_px_08', 'ask_px_08',
       'bid_sz_08', 'ask_sz_08', 'bid_ct_08', 'ask_ct_08', 'bid_px_09',
       'ask_px_09', 'bid_sz_09', 'ask_sz_09', 'bid_ct_09', 'a

In [39]:
df['symbol'].unique()

array(['AAPL'], dtype=object)

## Bid and Ask Order Flows

Let's compute the best-level OFI. As a reminder, here is the paragraph from the paper that gives the way of computing the Order Flow Imbalance

During the interval $[t - h, t]$, we enumerate the observations of all order book updates by $n$. Given two consecutive order book states for a given stock $i$ at $n-1$ and $n$, we compute the **bid order flows** ($\text{OF}^{m,b}_{i,n}$) and **ask order flows** ($\text{OF}^{m,a}_{i,n}$) of stock $i$ at level $m$ at time $n$ as:

$$
\text{OF}^{m,b}_{i,n} :=
\begin{cases}
q^{m,b}_{i,n}, & \text{if } P^{m,b}_{i,n} > P^{m,b}_{i,n-1} \\
q^{m,b}_{i,n} - q^{m,b}_{i,n-1}, & \text{if } P^{m,b}_{i,n} = P^{m,b}_{i,n-1} \\
- q^{m,b}_{i,n}, & \text{if } P^{m,b}_{i,n} < P^{m,b}_{i,n-1}
\end{cases}
$$

$$
\text{OF}^{m,a}_{i,n} :=
\begin{cases}
- q^{m,a}_{i,n}, & \text{if } P^{m,a}_{i,n} > P^{m,a}_{i,n-1} \\
q^{m,a}_{i,n} - q^{m,a}_{i,n-1}, & \text{if } P^{m,a}_{i,n} = P^{m,a}_{i,n-1} \\
q^{m,a}_{i,n}, & \text{if } P^{m,a}_{i,n} < P^{m,a}_{i,n-1}
\end{cases}
$$

**Where**:
- $P^{m,b}_{i,n}$, $q^{m,b}_{i,n}$: Bid price and size at level $m$ for stock $i$ at time $n$
- $P^{m,a}_{i,n}$, $q^{m,a}_{i,n}$: Ask price and size at level $m$ for stock $i$ at time $n$

### Interpretation:
- $\text{OF}^{m,b}_{i,t}$ is **positive** when:
  1. The bid price increases
  2. The bid price remains the same and the bid size increases
- It is **negative** when:
  1. The bid price decreases
  2. The bid price remains the same and the bid size decreases

An analogous interpretation holds for $\text{OF}^{m,a}_{i,t}$.


## Best-Level Order Flow Imbalance (OFI)

The **Best-Level OFI** calculates the cumulative order flow imbalances (OFIs) at the best bid and ask sides during a given time interval and is defined as:

$$
\text{OFI}^{1,h}_{i,t} := \sum_{n = N(t - h) + 1}^{N(t)} \left( \text{OF}^{1,b}_{i,n} - \text{OF}^{1,a}_{i,n} \right)
\tag{1}
$$

Where:
- $N(t - h) + 1$ and $N(t)$ are the indexes of the first and the last order book event in the interval $(t - h, t]$.
- $\text{OF}^{1,b}_{i,n}$ and $\text{OF}^{1,a}_{i,n}$ are the bid and ask order flows at the best level ($m=1$) for stock $i$ at time step $n$.

This metric helps quantify the net pressure from market participants at the best level during a specified time window.


**Order Flow Imbalance (OFI)** at the best level measures the net aggressive order flow at the best bid and ask. It captures how much buying vs selling pressure occurred by looking at changes in the best bid/ask volumes and prices. In formula form, for each stock i during a time window, best-level OFI is the sum of net volume changes at the best bid minus those at the best ask. We compute this by examining consecutive order book states:

- If the best bid price rises, a new buy order entered at a higher price, adding all its volume (positive bid flow).
- If the best bid price remains the same, the net bid flow is the change in bid size (increase = positive, decrease = negative).
- If the best bid price falls, the previous best bid was removed (either hit by a market sell or canceled), so we take a negative flow equal to the prior bid volume (since that volume left the bid side).

And symmetrically for the ask side:

- If the best ask price falls, a new sell order posted at a lower (better) price, adding all its volume (positive ask flow).
- If the best ask price remains the same, net ask flow is the change in ask size.
- If the best ask price rises, the previous best ask was lifted (bought) or canceled, so we take a negative flow equal to the prior ask volume.
Using these rules, we calculate the best-level OFI for each update event, then aggregate over each minute. The pseudocode below demonstrates this calculation for one stock:

In [40]:
#There is only data about Apple in the dataframe

symbol_data = df.sort_index()  # sort by time
# Initialize list to collect per-event OFI
ofi_events = []
prev_bid_price = None
prev_bid_size = None
prev_ask_price = None
prev_ask_size = None

for t, row in symbol_data.iterrows():
    # Skip first event (no previous state to compare)
    if prev_bid_price is None:
        prev_bid_price = row['bid_px_01']; prev_bid_size = row['bid_sz_01']
        prev_ask_price = row['ask_px_01']; prev_ask_size = row['ask_sz_01']
        continue
    # Bid-side order flow
    if row['bid_px_01'] > prev_bid_price:
        of_bid = row['bid_sz_01']         # new higher bid placed
    elif row['bid_px_01'] == prev_bid_price:
        of_bid = row['bid_sz_01'] - prev_bid_size  # size changed at same price
    else:
        of_bid = - prev_bid_size          # best bid fell, remove prev volume
    # Ask-side order flow
    if row['ask_px_01'] < prev_ask_price:
        of_ask = row['ask_sz_01']        # new lower ask placed
    elif row['ask_px_01'] == prev_ask_price:
        of_ask = row['ask_sz_01'] - prev_ask_size  # size changed at same price
    else:
        of_ask = - prev_ask_size          # best ask rose, remove prev volume
    # OFI = bid flow minus ask flow
    ofi_events.append({
        'symbol': row['symbol'],
        'timestamp': t,
        'OFI_best': of_bid - of_ask
    })
    # Update previous state for next loop
    prev_bid_price, prev_bid_size = row['bid_px_01'], row['bid_sz_01']
    prev_ask_price, prev_ask_size = row['ask_px_01'], row['ask_sz_01']


In [41]:
ofi_best_df = pd.DataFrame(ofi_events)
# Aggregate to 1-minute bins (summing OFI within each minute per symbol)
ofi_best_1min = ofi_best_df.groupby(['symbol', pd.Grouper(key='timestamp', freq='1Min')])['OFI_best'].sum().reset_index()
print(ofi_best_1min.head(5))

  symbol                 timestamp  OFI_best
0   AAPL 2024-10-21 11:54:00+00:00       600
1   AAPL 2024-10-21 11:55:00+00:00       974
2   AAPL 2024-10-21 11:56:00+00:00         0
3   AAPL 2024-10-21 11:57:00+00:00       399
4   AAPL 2024-10-21 11:58:00+00:00       931


This yields one OFI value per minute per stock, representing the net order imbalance at the top of the book in that minute. Positive OFI means net buyer-initiated volume (demand exceeds supply), while negative OFI means net selling pressure. Best-level OFI is known to be a major driver of short-term price changes.

### Deeper-Level OFI

From the paper, the deeper-level OFI is computed as below:

A natural extension of the best-level OFI defined in equation (1) is **deeper-level OFI** (see Xu *et al.* 2018, Kolm *et al.* 2023). We define OFI at level $m$ ($m \geq 1$) as follows:

$$
\text{OFI}^{m,h}_{i,t} := \sum_{n = N(t - h) + 1}^{N(t)} \left( \text{OF}^{m,b}_{i,n} - \text{OF}^{m,a}_{i,n} \right)
\tag{2}
$$

Due to the intraday pattern in limit order depth, we use the **average size** to scale OFIs at the corresponding levels (consistent with Ahn *et al.* 2001, Harris and Panchapagesan 2005), and consider:

$$
\text{ofi}^{m,h}_{i,t} = \frac{\text{OFI}^{m,h}_{i,t}}{\bar{Q}^{M,h}_{i,t}}
\tag{3}
$$

Where:

$$
\bar{Q}^{M,h}_{i,t} = \frac{1}{M} \sum_{m=1}^{M} \left( \frac{1}{2 \Delta N(t)} \sum_{n = N(t - h) + 1}^{N(t)} \left[ q^{m,b}_{i,n} + q^{m,a}_{i,n} \right] \right)
$$

- $\bar{Q}^{M,h}_{i,t}$ is the **average order book depth** across the first $M$ levels
- $\Delta N(t) = N(t) - N(t - h)$ is the number of events in the interval $(t - h, t]$
- In this paper, the top $M = 10$ levels of the limit order book (LOB) are used

Finally, we denote the **multi-level OFI vector** as:

$$
\text{ofi}^{(h)}_{i,t} = \left( \text{ofi}^{1,h}_{i,t}, \ldots, \text{ofi}^{10,h}_{i,t} \right)^\top
$$


Looking only at the best level may ignore significant activity deeper in the book. Large orders can sit just behind the best prices, and changes at those levels can also impact future price movements. To capture this, we extend the OFI calculation to multiple levels: For each depth level $ m = 1,\dots,10 $, we compute the order flow imbalance at that level $ (OFI{m}) $ using the same logic (comparing $ P{m} $ and $ q_{m} $ at time $ n $ vs $ n-1 $ for both bid and ask sides). For example, OFI at level 2 considers changes to the second-best bid and ask. We then aggregate each level’s OFI over the 1-minute interval:

In [42]:
# Compute multi-level OFI (levels 1-10) for each symbol and minute
ofi_multi_1min = []
for symbol, group in df.groupby('symbol'):
    # Sort events by time
    group = group.sort_index()
    # Initialize previous state (for levels 1-10)
    prev = None
    for t, row in group.iterrows():
        if prev is None:
            prev = row  # store previous row values for next iteration
            continue
        # Compute OFI for each level 1-10
        ofi_levels = {}
        for m in range(1, 10):
            bp, bv = row[f'bid_px_0{m}'], row[f'bid_sz_0{m}']
            ap, av = row[f'ask_px_0{m}'], row[f'ask_sz_0{m}']
            prev_bp, prev_bv = prev[f'bid_px_0{m}'], prev[f'bid_sz_0{m}']
            prev_ap, prev_av = prev[f'ask_px_0{m}'], prev[f'ask_sz_0{m}']
            # Bid side at level m
            if bp > prev_bp:
                of_bid = bv
            elif bp == prev_bp:
                of_bid = bv - prev_bv
            else:
                of_bid = - prev_bv
            # Ask side at level m
            if ap < prev_ap:
                of_ask = av
            elif ap == prev_ap:
                of_ask = av - prev_av
            else:
                of_ask = - prev_av
            ofi_levels[f'OFI_level{m}'] = of_bid - of_ask
        ofi_levels.update({'symbol': symbol, 'timestamp': t})
        ofi_multi_1min.append(ofi_levels)
        prev = row  # update previous snapshot
# Convert to DataFrame and aggregate 1-min sums per level
ofi_multi_df = pd.DataFrame(ofi_multi_1min)
ofi_multi_1min = ofi_multi_df.groupby(['symbol', pd.Grouper(key='timestamp', freq='1Min')]) \
                              .sum().reset_index()
print(ofi_multi_1min.filter(like='OFI_level').head(3))  # show first few OFI values


   OFI_level1  OFI_level2  OFI_level3  OFI_level4  OFI_level5  OFI_level6  \
0         600          10         -30          29         225         200   
1         974        1228         152          10         -84         -10   
2           0        1200           0          -4           0           0   

   OFI_level7  OFI_level8  OFI_level9  
0          44         155         400  
1        -110        -100         -55  
2           1           0           0  


In [43]:
# This is the dataframe the contains the multi-level OFI aggregated every minute
ofi_multi_1min

,symbol,timestamp,OFI_level1,OFI_level2,OFI_level3,OFI_level4,OFI_level5,OFI_level6,OFI_level7,OFI_level8,OFI_level9
0,AAPL,2024-10-21 11:54:00+00:00,600,10,-30,29,225,200,44,155,400
1,AAPL,2024-10-21 11:55:00+00:00,974,1228,152,10,-84,-10,-110,-100,-55
2,AAPL,2024-10-21 11:56:00+00:00,0,1200,0,-4,0,0,1,0,0
3,AAPL,2024-10-21 11:57:00+00:00,399,392,-121,-610,57,-219,-134,-98,-499
4,AAPL,2024-10-21 11:58:00+00:00,931,555,182,599,-587,241,134,188,444
...,...,...,...,...,...,...,...,...,...,...,...
66,AAPL,2024-10-21 13:00:00+00:00,-87,1532,-391,-1407,-1,-505,85,-124,0
67,AAPL,2024-10-21 13:01:00+00:00,-1264,-1977,-942,-692,-1169,270,286,1,585
68,AAPL,2024-10-21 13:02:00+00:00,-5658,-3880,-2461,-2189,-2226,-1538,-856,-390,-974
69,AAPL,2024-10-21 13:03:00+00:00,-1198,1128,842,643,737,611,1046,816,868


Now we have 10 OFI features per timestamp (one for each depth level 1–10). Deeper levels typically have larger resting volumes but may not change as frequently. To make them comparable, it is common to normalize the OFI at each level by the typical depth at that level. For example, if level 10 usually has 5x the volume of level 1, a raw OFI at level 10 of 500 shares might be less influential than an OFI of 100 shares at level 1. We scale each level’s OFI by the average order book size at that level (or use standard z-scores) to account for this depth effect:

In [44]:
# Scale OFI at each level by the average depth at that level (for each symbol)
for m in range(1, 10):
    col = f'OFI_level{m}'
    avg_depth_m = ofi_multi_1min.groupby('symbol')[col].transform('mean')  # average imbalance magnitude
    ofi_multi_1min[col] = ofi_multi_1min[col] / (avg_depth_m.replace(0, 1))

print(ofi_multi_1min.head(3))  # show first few OFI values

  symbol                 timestamp  OFI_level1  OFI_level2  OFI_level3  \
0   AAPL 2024-10-21 11:54:00+00:00   -1.978634    0.023126   -0.169101   
1   AAPL 2024-10-21 11:55:00+00:00   -3.211983    2.839815    0.856780   
2   AAPL 2024-10-21 11:56:00+00:00   -0.000000    2.775064    0.000000   

   OFI_level4  OFI_level5  OFI_level6  OFI_level7  OFI_level8  OFI_level9  
0   -0.775810   -3.601217  -19.777159    1.554229    5.402553    7.626208  
1   -0.267521    1.344454    0.988858   -3.885572   -3.485518   -1.048604  
2    0.107008   -0.000000   -0.000000    0.035323    0.000000    0.000000  


After scaling, each level’s OFI is on a comparable scale. The motivation for multi-level OFI is that order flow imbalances deeper in the book can also impact prices once the best quotes are depleted, and incorporating multiple levels gives a more complete picture of the buying/selling pressure. If we only looked at the best level, we might miss a large hidden iceberg order at level 2 or 3 that significantly affects price movements when the best level is consumed. Researchers have found that including multi-level order flow improves the explanatory power for price impact beyond just the top level.

## Integrated OFI

With 10 dimensions of OFI (one per level), it is helpful to summarize them into a single integrated metric. Integrated OFI combines the information from levels 1–10 into one feature. The approach by Cont et al. is to use Principal Component Analysis (PCA) on the vector of multi-level OFIs and use the first principal component as the integrated OFI.

 The first principal component captures the common imbalance across levels – essentially a weighted average of the OFIs at each depth that explains the most variance. We perform PCA for each stock on the scaled OFI levels 1–10. Before PCA, we ensure each level’s OFI series is standardized (mean 0, unit variance) so that no single level dominates due to scale. Then we extract the first principal component time series:

In [45]:
ofi_multi_1min

,symbol,timestamp,OFI_level1,OFI_level2,OFI_level3,OFI_level4,OFI_level5,OFI_level6,OFI_level7,OFI_level8,OFI_level9
0,AAPL,2024-10-21 11:54:00+00:00,-1.978634,0.023126,-0.169101,-0.775810,-3.601217,-19.777159,1.554229,5.402553,7.626208
1,AAPL,2024-10-21 11:55:00+00:00,-3.211983,2.839815,0.856780,-0.267521,1.344454,0.988858,-3.885572,-3.485518,-1.048604
2,AAPL,2024-10-21 11:56:00+00:00,-0.000000,2.775064,0.000000,0.107008,-0.000000,-0.000000,0.035323,0.000000,0.000000
3,AAPL,2024-10-21 11:57:00+00:00,-1.315792,0.906521,-0.682042,16.318764,-0.912308,21.655989,-4.733333,-3.415808,-9.513695
4,AAPL,2024-10-21 11:58:00+00:00,-3.070181,1.283467,1.025881,-16.024491,9.395176,-23.831476,4.733333,6.552774,8.465091
...,...,...,...,...,...,...,...,...,...,...,...
66,AAPL,2024-10-21 13:00:00+00:00,0.286902,3.542831,-2.203954,37.640166,0.016005,49.937326,3.002488,-4.322042,0.000000
67,AAPL,2024-10-21 13:01:00+00:00,4.168323,-4.571917,-5.309781,18.512434,18.710325,-26.699164,10.102488,0.034855,11.153330
68,AAPL,2024-10-21 13:02:00+00:00,18.658523,-8.972705,-13.871943,58.560286,35.628043,152.086351,-30.236816,-13.593520,-18.569817
69,AAPL,2024-10-21 13:03:00+00:00,3.950673,2.608560,4.746110,-17.201583,-11.795987,-60.419220,36.948259,28.441826,16.548872


In [46]:
from sklearn.decomposition import PCA

integrated_ofi_list = []
for symbol, group in ofi_multi_1min.groupby('symbol'):
    # Take the multi-level OFI columns for this symbol
    X = group[[f'OFI_level{m}' for m in range(1, 10)]].values  
    # Standardize features (mean 0, std 1)
    X = (X - X.mean(axis=0)) / X.std(axis=0)
    # Fit PCA on 10-dimensional OFI
    pca = PCA(n_components=9)
    X_pca = pca.fit_transform(X)
    # First principal component time series
    integrated_ofi = X_pca[:, 0]
    # Store integrated OFI with corresponding timestamps
    out = pd.DataFrame({
        'symbol': symbol,
        'timestamp': group['timestamp'],
        'OFI_integrated': integrated_ofi
    })
    integrated_ofi_list.append(out)
integrated_ofi_df = pd.concat(integrated_ofi_list)
print(integrated_ofi_df.head(3))


  symbol                 timestamp  OFI_integrated
0   AAPL 2024-10-21 11:54:00+00:00       -0.709793
1   AAPL 2024-10-21 11:55:00+00:00       -0.157859
2   AAPL 2024-10-21 11:56:00+00:00       -0.175088


#### The first principal component typically explains a large share of the variance in the multi-level OFI (because order imbalances often co-move across levels – e.g. a strong buy imbalance might clear out several levels on the ask side).

## 4. Cross-Asset OFI (Sparse Cross-Impact via Lasso Regression)

So far, we have treated each stock independently. However, order flow in one stock can impact the prices of other stocks – this is what the paper calls **cross-impact**. For example, heavy buying in a large tech stock might coincide with price upticks in other tech stocks. 

To quantify this, we consider **Cross-Asset OFI features**: for a given stock, we include the OFIs of *other* stocks as potential predictors of its return.

A straightforward model for contemporaneous price impact is a linear regression of a stock’s return $r_i(t)$ on its own OFI and others’ OFIs at the same time:

$$
r_i(t) = \beta_{i,i} \cdot \text{OFI}_i(t) + \sum_{j \ne i} \beta_{i,j} \cdot \text{OFI}_j(t) + \epsilon_i(t)
$$

Including all cross-asset terms (for potentially dozens of stocks) can lead to **overfitting** and irrelevant coefficients. The authors address this by using **Lasso regression**, which adds an $L_1$ penalty to induce sparsity — effectively selecting only the most influential cross-asset OFIs. 

This means the model will zero out many $\beta_{i,j}$ coefficients if those OFIs don’t help explain stock $i$'s returns, resulting in a **sparse cross-impact model**.

---

We construct a **feature matrix** where each column is a stock’s integrated OFI (or best-level OFI, depending on the model) for each minute, and the **response** is the vector of stock returns.

**Unfortunately the dataset provided only shows data for the stock AAPL, and no other stock, so it is impossible for me to run the cross asset regression. I have tried finding online data, but I couldn't get this level of granularity (only daily data). To show how I would perform the code, I would just make up some fictional data from the AAPL time-series and then run the cross-sectional model. Here is still a placeholder that would work if I had additional data**


In [ ]:
from sklearn.linear_model import LassoCV

# Prepare design matrix X (columns: OFI_integrated of AAA, BBB, CCC) and target y (returns of AAA)
X = integrated_ofi_df.pivot(index='timestamp', columns='symbol', values='OFI_integrated').values
y = returns_df[returns_df['symbol']=='AAPL']['1min_return'].values  # assuming we have returns data

# Fit Lasso with cross-validation to find optimal regularization
lasso = LassoCV(cv=5, random_state=0).fit(X, y)
print("Selected alpha:", lasso.alpha_)
coef = pd.Series(lasso.coef_, index=sorted(df['symbol'].unique()))
print(coef)


NameError: name 'returns_df' is not defined

#### If you send me more data, I would be happy to run this regression and show the predictive power of this model ! I would also like to try other Machine Learning algorithms that would predict well the future returns while maintaining sparsity.